# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 1

if debug:
    print(debug , incremental_run)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get receipts from: {filterdate}')

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 7, Finished, Available, Finished)

Get receipts from: 2025-05-08


# Init Query(s)

In [7]:
receipts_1_df = spark.sql("""
SELECT
     '1' AS Record_Type
    ,'ENDCTG' AS Company_Code
    ,whsloadtable.loadid AS Receipt_Reference
    ,CASE WHEN DATE(whsloadtable.loadshipconfirmutcdatetime) = '1900-01-01' THEN NULL ELSE DATE(whsloadtable.loadshipconfirmutcdatetime) END AS Receipt_Date
    ,'PEIM' AS Receipt_Type
    ,hslcommercialinvoice.portairportofarrival AS Port_Airport_Of_Arrival
    ,hslcommercialinvoice.airportofdeparture AS Airport_Of_Departure
    ,MIN(hslcommercialinvoice.nationality) AS Nationality
    ,'' AS Inland_Depot
    ,'' AS Container_ID
    ,INT(MIN(hslcommercialinvoice.modeoftransport)) AS Mode_Of_Transport
    ,3 AS Inland_Mode_Of_Transport
    ,CAST(SUM(hslcommercialinvoice.value) AS DECIMAL(12 , 4)) AS Total_Value
    ,INT(MIN(hslcommercialinvoice.numberofpackages)) AS Number_Of_Packages
    ,'' AS Volume_Cubic_Metres
    ,CAST(SUM(hslcommercialinvoice.grossweight) AS DECIMAL(10 , 4)) AS Gross_Weight
    ,'' AS Agent
    ,'' AS Carrier
    ,CAST(MIN(hslcommercialinvoice.freightcharges) AS DECIMAL(10 , 4)) AS Freight
    ,MIN(hslcommercialinvoice.freightchargescurrency) AS Freight_Currency
    ,MIN(hslcommercialinvoice.freightbasis) AS Freight_Basis
    ,'' AS Air_Freight
    ,'' AS Air_Freight_Currency
    ,'' AS Air_Freight_Basis
    ,CASE WHEN DATE(hslcommercialinvoice.shippeddate) = '1900-01-01' THEN NULL ELSE DATE(hslcommercialinvoice.shippeddate) END AS Shipped_Date
    ,'' AS Date_Of_Arrival
    ,'' AS Voyage_Reference
    ,'' AS Ships_Name
    ,'' AS Post_Importation_Charges
    ,'' AS Post_Importation_Currency
    ,'' AS Post_Importation_Basis
    ,'' AS Container_Number
    ,INT(MIN(hslcommercialinvoice.arrivaltransportidtype)) AS Arrival_Transport_ID_Type
    ,STRING(MIN(hslcommercialinvoice.arrivaltransportid)) AS Arrival_Transport_ID
    ,MIN(hslcommercialinvoice.mrn) AS MRN
    ,'' AS Previous_Document_Ref
    ,'' AS Previous_Document_Type
    ,hslcommercialinvoice.bookingreference AS Key
    ,hslcommercialinvoice.bookingreference AS Level_Key
    ,'' AS Parent_Key

FROM hslcommercialinvoice

LEFT JOIN whsloadtable
    ON hslcommercialinvoice.bookingreference = whsloadtable.hslcomminvbookingreference
    AND hslcommercialinvoice.dataareaid = whsloadtable.dataareaid

WHERE
    whsloadtable.hslcomminvbookingreference != ''
    AND whsloadtable.dataareaid IN ('end.', 'END.')
    AND whsloadtable.loadstatus != '0'

GROUP BY
     whsloadtable.loadid
    ,DATE(whsloadtable.loadshipconfirmutcdatetime)
    ,hslcommercialinvoice.portairportofarrival
    ,hslcommercialinvoice.airportofdeparture
    ,hslcommercialinvoice.bookingreference
    ,hslcommercialinvoice.shippeddate
""")

if debug:
    display(receipts_1_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7f9bb603-e7f0-48bd-8c54-c4943995f72d)

In [8]:
receipts_1_df = receipts_1_df.select(
    substring(col("Record_Type").cast("string"),1, 1).alias("Record_Type"),
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Receipt_Reference").cast("string"),1, 10).alias("Receipt_Reference"),
    col("Receipt_Date").cast("date").alias("Receipt_Date"),
    substring(col("Receipt_Type").cast("string"),1, 4).alias("Receipt_Type"),
    substring(col("Port_Airport_Of_Arrival").cast("string"),1, 6).alias("Port_Airport_Of_Arrival"),
    substring(col("Airport_Of_Departure").cast("string"),1, 4).alias("Airport_Of_Departure"),
    substring(col("Nationality").cast("string"),1, 4).alias("Nationality"),
    col("Inland_Depot").cast("string").alias("Inland_Depot"),
    col("Container_ID").cast("string").alias("Container_ID"),
    substring(col("Mode_Of_Transport").cast("string"),1, 2).alias("Mode_Of_Transport"),
    substring(col("Inland_Mode_Of_Transport").cast("string"),1, 2).alias("Inland_Mode_Of_Transport"),
    substring(col("Total_Value").cast("string"),1, 10).alias("Total_Value"),
    col("Number_Of_Packages").cast("string").alias("Number_Of_Packages"),
    col("Volume_Cubic_Metres").cast("string").alias("Volume_Cubic_Metres"),
    substring(col("Gross_Weight").cast("string"),1, 11).alias("Gross_Weight"),
    col("Agent").cast("string").alias("Agent"),
    col("Carrier").cast("string").alias("Carrier"),
    substring(col("Freight").cast("string"),1, 11).alias("Freight"),
    substring(col("Freight_Currency").cast("string"),1, 4).alias("Freight_Currency"),
    substring(col("Freight_Basis").cast("string"),1, 2).alias("Freight_Basis"),
    col("Air_Freight").cast("string").alias("Air_Freight"),
    col("Air_Freight_Currency").cast("string").alias("Air_Freight_Currency"),
    col("Air_Freight_Basis").cast("string").alias("Air_Freight_Basis"),
    col("Shipped_Date").cast("date").alias("Shipped_Date"),
    col("Date_Of_Arrival").cast("string").alias("Date_Of_Arrival"),
    col("Voyage_Reference").cast("string").alias("Voyage_Reference"),
    col("Ships_Name").cast("string").alias("Ships_Name"),
    col("Post_Importation_Charges").cast("string").alias("Post_Importation_Charges"),
    col("Post_Importation_Currency").cast("string").alias("Post_Importation_Currency"),
    col("Post_Importation_Basis").cast("string").alias("Post_Importation_Basis"),
    col("Container_Number").cast("string").alias("Container_Number"),
    substring(col("Arrival_Transport_ID_Type").cast("string"),1, 2).alias("Arrival_Transport_ID_Type"),
    substring(col("Arrival_Transport_ID").cast("string"),1, 35).alias("Arrival_Transport_ID"),
    substring(col("MRN").cast("string"),1, 18).alias("MRN"),
    col("Previous_Document_Ref").cast("string").alias("Previous_Document_Ref"),
    col("Previous_Document_Type").cast("string").alias("Previous_Document_Type"),
    col("Key").cast("string").alias("Key"),
    col("Level_Key").cast("string").alias("Level_Key"),
    col("Parent_Key").cast("string").alias("Parent_Key")
)
if debug:
    display(receipts_1_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a600ef5a-264a-40cd-820a-cb25390052ac)

In [9]:
receipts_2_df = spark.sql("""
SELECT
     '2' AS Record_Type
    ,'ENDCTG' AS Company_Code
    ,whsloadtable.loadid AS Receipt_Reference
    ,CASE WHEN DATE(whsloadtable.loadshipconfirmutcdatetime) = '1900-01-01' THEN NULL ELSE DATE(whsloadtable.loadshipconfirmutcdatetime) END AS Receipt_Date
    ,'PEIM' AS Receipt_Type
    ,concat_ws('-', purchtable.orderaccount, hslcommercialinvoice.commercialinvoiceid) AS Invoice_Key
    ,'' AS Invoice_Reference
    ,min(purchtable.orderaccount) AS Supplier_Reference
    ,MIN(dirpartytable.name) AS Supplier_Name
    ,'' AS Invoice_Charges
    ,CAST(SUM(hslcommercialinvoice.value) AS DECIMAL(12 , 4)) AS Invoice_Value
    ,MIN(hslcommercialinvoice.currency) AS Invoice_Currency
    ,MIN(hslcommercialinvoice.citermsduty) AS Trade_Terms
    ,'' AS Cash_Discount
    ,'' AS Invoice_Net_Weight
    ,'' AS Weight_Code
    ,'' AS Freight_Charges
    ,'' AS Freight_Charges_Currency
    ,'' AS Basis_of_Freight
    ,'' AS Air_Freight_Charges
    ,'' AS Air_Freight_Charges_Currency
    ,'' AS Basic_of_Air_Freight
    ,CAST(MIN(hslcommercialinvoice.insurancecharges) AS DECIMAL(10 , 4)) AS Insurance_Charges
    ,MIN(hslcommercialinvoice.insurancechargescurrency) AS Insurance_Charges_Currency
    ,'' AS Commission_Value
    ,'' AS Commission_Currency
    ,'' AS Basis_of_Commission
    ,'' AS Post_Importation_Charges
    ,'' AS Post_Importation_Currency
    ,'' AS Post_Importation_Basis
    ,'' AS Container_Number
    ,'' AS Gross_Weight
    ,'' AS Seller_Code
    ,MIN(hslcommercialinvoice.locationname) AS Location_Name
    ,CASE WHEN purchtable.purchasetype = 3 THEN '1' 
          WHEN purchtable.purchasetype = 4 THEN '2' 
          ELSE '1' 
     END AS `NOTC(a)`

    ,'1' AS `NOTC(b)`
    ,'' AS Country_of_Consignment
    ,concat_ws('||', hslcommercialinvoice.bookingreference, hslcommercialinvoice.commercialinvoiceid) AS Key
    ,hslcommercialinvoice.commercialinvoiceid AS Level_Key
    ,hslcommercialinvoice.bookingreference AS Parent_Key

FROM hslcommercialinvoice

LEFT JOIN whsloadtable
    ON hslcommercialinvoice.bookingreference = whsloadtable.hslcomminvbookingreference
    AND hslcommercialinvoice.dataareaid = whsloadtable.dataareaid

LEFT JOIN purchtable
    ON hslcommercialinvoice.purchaseorder = purchtable.purchid
    AND hslcommercialinvoice.dataareaid = purchtable.dataareaid

LEFT JOIN vendtable
    ON purchtable.orderaccount = vendtable.accountnum
    AND purchtable.dataareaid = vendtable.dataareaid

LEFT JOIN dirpartytable
    ON vendtable.party = dirpartytable.recid
    AND vendtable.dataareaid = dirpartytable.dataareaid

WHERE whsloadtable.hslcomminvbookingreference != ''
  AND whsloadtable.dataareaid IN ('end.', 'END.')
  AND whsloadtable.loadstatus != 0

GROUP BY
     whsloadtable.loadid
    ,DATE(whsloadtable.loadshipconfirmutcdatetime)
    ,concat_ws('-', purchtable.orderaccount, hslcommercialinvoice.commercialinvoiceid)
    ,CASE WHEN purchtable.purchasetype = 3 THEN '1' 
          WHEN purchtable.purchasetype = 4 THEN '2' 
          ELSE '1' 
     END
    ,concat_ws('||', hslcommercialinvoice.bookingreference, hslcommercialinvoice.commercialinvoiceid)
    ,hslcommercialinvoice.commercialinvoiceid
    ,hslcommercialinvoice.bookingreference
""")

if debug:
    display(receipts_2_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f2f1f53c-8879-413f-87f8-b1ec4ca0dc3e)

In [10]:
receipts_2_df = receipts_2_df.select(
    substring(col("Record_Type").cast("string"),1, 1).alias("Record_Type"),
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Receipt_Reference").cast("string"),1, 10).alias("Receipt_Reference"),
    col("Receipt_Date").cast("date").alias("Receipt_Date"),
    substring(col("Receipt_Type").cast("string"),1, 4).alias("Receipt_Type"),
    col("Invoice_Key").cast("string").alias("Invoice_Key"),
    substring(col("Invoice_Reference").cast("string"),1, 20).alias("Invoice_Reference"),
    substring(col("Supplier_Reference").cast("string"),1, 15).alias("Supplier_Reference"),
    substring(col("Supplier_Name").cast("string"),1, 25).alias("Supplier_Name"),
    substring(col("Invoice_Charges").cast("string"),1, 11).alias("Invoice_Charges"),
    substring(col("Invoice_Value").cast("string"),1, 10).alias("Invoice_Value"),
    substring(col("Invoice_Currency").cast("string"),1, 4).alias("Invoice_Currency"),
    substring(col("Trade_Terms").cast("string"),1, 1).alias("Trade_Terms"),
    col("Cash_Discount").cast("string").alias("Cash_Discount"),
    col("Invoice_Net_Weight").cast("string").alias("Invoice_Net_Weight"),
    col("Weight_Code").cast("string").alias("Weight_Code"),
    col("Freight_Charges").cast("string").alias("Freight_Charges"),
    col("Freight_Charges_Currency").cast("string").alias("Freight_Charges_Currency"),
    col("Basis_of_Freight").cast("string").alias("Basis_of_Freight"),
    col("Air_Freight_Charges").cast("string").alias("Air_Freight_Charges"),
    col("Air_Freight_Charges_Currency").cast("string").alias("Air_Freight_Charges_Currency"),
    col("Basic_of_Air_Freight").cast("string").alias("Basic_of_Air_Freight"),
    substring(col("Insurance_Charges").cast("string"),1, 11).alias("Insurance_Charges"),
    substring(col("Insurance_Charges_Currency"),1, 4).cast("string").alias("Insurance_Charges_Currency"),
    col("Commission_Value").cast("string").alias("Commission_Value"),
    col("Commission_Currency").cast("string").alias("Commission_Currency"),
    col("Basis_of_Commission").cast("string").alias("Basis_of_Commission"),
    col("Post_Importation_Charges").cast("string").alias("Post_Importation_Charges"),
    col("Post_Importation_Currency").cast("string").alias("Post_Importation_Currency"),
    col("Post_Importation_Basis").cast("string").alias("Post_Importation_Basis"),
    col("Container_Number").cast("string").alias("Container_Number"),
    col("Gross_Weight").cast("string").alias("Gross_Weight"),
    col("Seller_Code").cast("string").alias("Seller_Code"),
    substring(col("Location_Name").cast("string"),1, 37).alias("Location_Name"),
    substring(col("NOTC(a)").cast("string"),1, 1).alias("NOTC(a)"),
    substring(col("NOTC(b)").cast("string"),1, 1).alias("NOTC(b)"),
    substring(col("Country_of_Consignment").cast("string"),1, 1).alias("Country_of_Consignment"),
    col("Key").cast("string").alias("Key"),
    col("Level_Key").cast("string").alias("Level_Key"),
    col("Parent_Key").cast("string").alias("Parent_Key")
)
if debug:
    display(receipts_2_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, cde4fba5-a1d0-491b-a4e1-df6948f9349a)

In [11]:
receipts_3_df = spark.sql("""
SELECT
     '3' AS Record_Type
    ,'ENDCTG' AS Company_Code
    --,'' AS Site_Code >>>> REMOVED AS MESSES UP THE ORDERING - ADD EXTRA PIPE LATER
    ,whsloadtable.loadid AS Receipt_Reference
    ,CASE WHEN DATE(whsloadtable.loadshipconfirmutcdatetime) = '1900-01-01' THEN NULL ELSE DATE(whsloadtable.loadshipconfirmutcdatetime) END AS Receipt_Date
    ,'PEIM' AS Receipt_Type
    ,concat_ws('-', purchtable.orderaccount, hslcommercialinvoice.commercialinvoiceid) AS Invoice_Key
    ,hslcommercialinvoice.itemnumber AS Product_Code

    ,concat_ws(
        '', 
        ecoresproducttranslation.name,
        inventtable.endfabriccomposition,
        CASE 
            WHEN inventtable.endmenswear = 1 AND inventtable.endwomenswear = 1 THEN 'U'
            WHEN inventtable.endmenswear = 1 THEN 'M'
            WHEN inventtable.endwomenswear = 1 THEN 'F'
            ELSE ''
        END
    ) AS Product_Description

    ,STRING(hslcommercialinvoice.commoditycode) AS Product_Tariff_Commodity_Code
    ,vatgrouplookup.LangdonCode AS Product_VAT_Rate_Identifier
    ,'' AS Product_Unit_Weight
    ,'BOND' AS Project_Reference
    ,'' AS IPR_Reference
    ,'' AS OPR_Project_Reference
    ,purchtable.purchid AS Order_Reference
    ,hslcommercialinvoice.countryoforigin AS Country_of_Origin
    ,hslcommercialinvoice.dispatchcountry AS Country_of_Consignment
    ,CAST(hslcommercialinvoice.invoicedqty AS DECIMAL(10 , 3)) AS Invoiced_Quantity
    ,ecorescategoryintrastat.additionalunits AS Quantity_Code
    ,CAST(hslcommercialinvoice.polinetotal AS DECIMAL(10 , 3)) AS Item_Value
    ,CAST(hslcommercialinvoice.invoicedqty AS DECIMAL(10 , 3)) AS Received_Quantity
    ,'' AS Licence_Reference
    ,IFNULL(hslcommercialinvoice.preferencedocumenttype, 'G') AS Preference_Document_Type -- <<<< this temp solution, but should be this >>>>     ,hslcommercialinvoice.preferencedocumenttype AS Preference_Document_Type
    ,hslcommercialinvoice.preferencedocumentreference AS Preference_Document_Reference
    ,'' AS Item_Net_Weight
    ,'' AS Container_Number
    ,'' AS Package_Count
    ,'' AS Package_Kind
    ,'' AS Package_Marks_And_Numbers
    ,'' AS Gross_Weight
    ,'' AS Seller_Code
    ,hslcommercialinvoice.preferencecountry AS Preference
    ,hslcommercialinvoice.preferencegroup AS Preference_Group
    ,'' AS `NOTC(a)`
    ,'' AS `NOTC(b)`
    ,hslcommercialinvoice.preferencedocumentcode AS Preference_Document_Code
    ,hslcommercialinvoice.preferencedocumentstatuscode AS Preference_Document_Status_Code
    ,concat_ws('||', hslcommercialinvoice.bookingreference, hslcommercialinvoice.commercialinvoiceid, hslcommercialinvoice.recid) AS Key
    ,hslcommercialinvoice.recid AS Level_Key
    ,hslcommercialinvoice.commercialinvoiceid AS Parent_Key

FROM hslcommercialinvoice

LEFT JOIN whsloadtable
    ON hslcommercialinvoice.bookingreference = whsloadtable.hslcomminvbookingreference
    AND hslcommercialinvoice.dataareaid = whsloadtable.dataareaid

LEFT JOIN purchtable
    ON hslcommercialinvoice.purchaseorder = purchtable.purchid
    AND hslcommercialinvoice.dataareaid = purchtable.dataareaid

LEFT JOIN inventtable
    ON hslcommercialinvoice.itemnumber = inventtable.itemid
    AND hslcommercialinvoice.dataareaid = inventtable.dataareaid

LEFT JOIN ecoresproducttranslation
    ON inventtable.product = ecoresproducttranslation.product

LEFT JOIN inventtablemodule
    ON inventtable.itemid = inventtablemodule.itemid
    AND inventtable.dataareaid = inventtablemodule.dataareaid
    AND inventtablemodule.moduletype = 1

LEFT JOIN vatgrouplookup
    ON inventtablemodule.taxitemgroupid = vatgrouplookup.ItemVATgroup

LEFT JOIN ecorescategoryintrastat
    ON inventtable.intrastatcommodity = ecorescategoryintrastat.category

WHERE
    whsloadtable.hslcomminvbookingreference != ''
    AND whsloadtable.dataareaid IN ('end.', 'END.')
    AND whsloadtable.loadstatus != '0'
""")

if debug:
    display(receipts_3_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 8ff13acb-5ea7-4b02-b90a-c9e026eb330a)

In [12]:
receipts_3_df = receipts_3_df.select(
    substring(col("Record_Type").cast("string"),1, 1).alias("Record_Type"),
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    #substring(col("Site_Code).cast("string"),1, 4).alias("Site_Code"), >>>> REMOVED AS MESSES UP THE ORDERING - ADD EXTRA PIPE LATER - Put this back in when it's back in
    substring(col("Receipt_Reference").cast("string"),1, 10).alias("Receipt_Reference"),
    col("Receipt_Date").cast("date").alias("Receipt_Date"),
    substring(col("Receipt_Type").cast("string"),1, 4).alias("Receipt_Type"),
    substring(col("Invoice_Key").cast("string"),1, 15).alias("Invoice_Key"),
    substring(col("Product_Code").cast("string"),1, 25).alias("Product_Code"),
    substring(col("Product_Description").cast("string"),1, 120).alias("Product_Description"),
    substring(col("Product_Tariff_Commodity_Code").cast("string"),1, 10).alias("Product_Tariff_Commodity_Code"),
    substring(col("Product_VAT_Rate_Identifier").cast("string"),1, 1).alias("Product_VAT_Rate_Identifier"),
    col("Product_Unit_Weight").cast("string").alias("Product_Unit_Weight"),
    substring(col("Project_Reference").cast("string"),1, 10).alias("Project_Reference"),
    col("IPR_Reference").cast("string").alias("IPR_Reference"),
    col("OPR_Project_Reference").cast("string").alias("OPR_Project_Reference"),
    substring(col("Order_Reference").cast("string"),1, 10).alias("Order_Reference"),
    substring(col("Country_of_Origin").cast("string"),1, 4).alias("Country_of_Origin"),
    substring(col("Country_of_Consignment").cast("string"),1, 4).alias("Country_of_Consignment"),
    substring(col("Invoiced_Quantity").cast("string"),1, 11).alias("Invoiced_Quantity"),
    substring(col("Quantity_Code").cast("string"),1, 3).alias("Quantity_Code"),
    substring(col("Item_Value").cast("string"),1, 10).alias("Item_Value"),
    substring(col("Received_Quantity").cast("string"),1, 11).alias("Received_Quantity"),
    col("Licence_Reference").cast("string").alias("Licence_Reference"),
    substring(col("Preference_Document_Type"),1, 1).cast("string").alias("Preference_Document_Type"),
    substring(col("Preference_Document_Reference"),1, 10).cast("string").alias("Preference_Document_Reference"),
    col("Item_Net_Weight").cast("string").alias("Item_Net_Weight"),
    col("Container_Number").cast("string").alias("Container_Number"),
    col("Package_Count").cast("string").alias("Package_Count"),
    col("Package_Kind").cast("string").alias("Package_Kind"),
    col("Package_Marks_And_Numbers").cast("string").alias("Package_Marks_And_Numbers"),
    col("Gross_Weight").cast("string").alias("Gross_Weight"),
    col("Seller_Code").cast("string").alias("Seller_Code"),
    col("Preference").cast("string").alias("Preference"),
    col("Preference_Group").cast("string").alias("Preference_Group"),
    col("NOTC(a)").cast("string").alias("NOTC(a)"),
    col("NOTC(b)").cast("string").alias("NOTC(b)"),
    substring(col("Preference_Document_Code").cast("string"),1, 4).alias("Preference_Document_Code"),
    substring(col("Preference_Document_Status_Code").cast("string"),1, 4).alias("Preference_Document_Status_Code"),
    col("Key").cast("string").alias("Key"),
    col("Level_Key").cast("string").alias("Level_Key"),
    col("Parent_Key").cast("string").alias("Parent_Key")
)
if debug:
    display(receipts_3_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 666617f0-674f-43ff-ba90-4f28070927ea)

## Date Field Changing Post Query(s)

In [13]:
list_date_columns_1 = [name for name, dtype in receipts_1_df.dtypes if dtype in ('date','timestamp')]
list_date_columns_2 = [name for name, dtype in receipts_2_df.dtypes if dtype in ('date','timestamp')]
list_date_columns_3 = [name for name, dtype in receipts_3_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)
    print("Date Columns to change: " , list_date_columns_2)
    print("Date Columns to change: " , list_date_columns_3)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 14, Finished, Available, Finished)

Date Columns to change:  ['Receipt_Date', 'Shipped_Date']
Date Columns to change:  ['Receipt_Date']
Date Columns to change:  ['Receipt_Date']


In [14]:
for column in list_date_columns_1:
    receipts_1_df = receipts_1_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

for column in list_date_columns_2:
    receipts_2_df = receipts_2_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

for column in list_date_columns_3:
    receipts_3_df = receipts_3_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 15, Finished, Available, Finished)

## Display Pre-Filtering

In [15]:
if debug:
    display(receipts_1_df)
    display(receipts_2_df)
    display(receipts_3_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 16, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6306c5a3-6abf-4c48-becf-835d8daf9a97)

SynapseWidget(Synapse.DataFrame, 60fea3a5-10ae-45fd-9aac-a7cba705d7a2)

SynapseWidget(Synapse.DataFrame, f95090a4-d655-4a1c-bc7c-04f008617959)

# Init Error Check Process Level 1

In [16]:
receipts_1_Mandatory_Columns = [
    "Record_Type",
    "Company_Code",
    "Receipt_Reference",
    "Receipt_Date",
    "Receipt_Type",
    "Port_Airport_Of_Arrival",
    # "Airport_of_Departure",
    "Nationality",
    "Mode_of_Transport",
    "Inland_Mode_of_Transport",
    "Total_Value",
    "Number_Of_Packages",
    "Shipped_Date",
    "Date_of_Arrival"
]

receipts_1_Cant_Be_Zero_Columns = [
    "Total_Value",
    "Number_Of_Packages",
    "Gross_Weight"
]

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 17, Finished, Available, Finished)

In [17]:
level = 1

null_condition = None
null_column_names_exprs = []

zero_condition = None
zero_column_names_exprs = []


for column in receipts_1_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

for column in receipts_1_Cant_Be_Zero_Columns:
    if column in ["Total_Value", "Gross_Weight"]:
        condition = col(column) == "0.0000"
    elif column == "Number_Of_Packages":
        condition = col(column) == "0"
    zero_condition = condition if zero_condition is None else zero_condition | condition
    zero_column_names_exprs.append(when(condition, lit(column)))


# Collect error columns into arrays
receipts_1_with_errors = receipts_1_df.withColumn("null_failed_columns", array(*null_column_names_exprs)) \
                                      .withColumn("zero_failed_columns", array(*zero_column_names_exprs))

# Filter out nulls from those arrays
receipts_1_with_errors = receipts_1_with_errors.withColumn(
    "null_failed_columns", expr("filter(null_failed_columns, x -> x is not null)")
).withColumn(
    "zero_failed_columns", expr("filter(zero_failed_columns, x -> x is not null)")
)

# Generate the error messages (only when columns exist)
receipts_1_with_errors = receipts_1_with_errors.withColumn(
    "null_errors",
    when(size(col("null_failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(" , ", col("null_failed_columns")), lit(f" are null at level {level}")))
).withColumn(
    "zero_errors",
    when(size(col("zero_failed_columns")) > 0,
         concat_ws("", concat_ws(", ", col("zero_failed_columns")), lit(f" are 0 at level {level}")))
)

# Combine all errors
receipts_1_with_errors = receipts_1_with_errors.withColumn(
    "error_fields",
    concat_ws(" , ", col("null_errors"), col("zero_errors"))
)

# Filter bad and good
receipts_1_bad_df = receipts_1_with_errors.filter(null_condition | zero_condition) \
    .drop("null_errors", "zero_errors", "null_failed_columns", "zero_failed_columns")

receipts_1_df = receipts_1_with_errors.filter(~(null_condition | zero_condition)) \
    .drop("error_fields", "null_errors", "zero_errors", "null_failed_columns", "zero_failed_columns")

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 18, Finished, Available, Finished)

In [18]:
if debug:
    display(receipts_1_bad_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 19, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f2918b31-4f84-42b1-a3bc-12e4144d1276)

# Init Error Check Process Level 2

In [19]:
receipts_2_Mandatory_Columns = [
    "Receipt_Date",
    "Receipt_Reference",
    "Invoice_Key",
    "Record_Type",
    "`NOTC(a)`",
    "Trade_Terms",
    "Location_Name",
    "Supplier_Reference",
    "Receipt_Type",
    "Invoice_Currency",
    "Invoice_Value",
    "Company_Code"
  ]

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 20, Finished, Available, Finished)

In [20]:
level = 2

null_condition = None
null_column_names_exprs = []

# Check for null values in the mandatory columns
for column in receipts_2_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Perform the join with level 1 and ensure the columns are in the right order
receipts_2_with_join = receipts_2_df.join(
    receipts_1_bad_df.select("Receipt_Reference", "error_fields").withColumnRenamed("error_fields", "level1_errors"),
    on="Receipt_Reference",
    how="left"
)

# Build array of failed columns for level 2
receipts_2_with_errors = receipts_2_with_join.withColumn(
    "level2_failed_columns", array(*null_column_names_exprs)
)

# Remove nulls from failed columns array
receipts_2_with_errors = receipts_2_with_errors.withColumn(
    "level2_nonnull_failed_columns",
    expr("filter(level2_failed_columns, x -> x is not null)")
)

# Create level 2 error message
receipts_2_with_errors = receipts_2_with_errors.withColumn(
    "level2_error_fields",
    when(
        size(col("level2_nonnull_failed_columns")) > 0,
        concat_ws("", lit("Columns "), concat_ws(", ", col("level2_nonnull_failed_columns")), lit(f" are null at level {level}"))
    ).otherwise(lit(""))
)

# Combine level 1 and level 2 error messages
receipts_2_with_errors = receipts_2_with_errors.withColumn(
    "error_fields",
    concat_ws("; ",
        *[col(c) for c in ["level1_errors", "level2_error_fields"] if c in receipts_2_with_errors.columns]
    )
)

# Split into bad and good rows
receipts_2_bad_df = receipts_2_with_errors.filter(
    null_condition | col("level1_errors").isNotNull()
).drop("level1_errors", "level2_failed_columns", "level2_nonnull_failed_columns", "level2_error_fields")

# Ensure good rows have the same column order as bad rows
receipts_2_df = receipts_2_with_errors.filter(
    ~(null_condition | col("level1_errors").isNotNull())
).drop("level1_errors", "level2_failed_columns", "level2_nonnull_failed_columns", "level2_error_fields", "error_fields")

# Explicitly reorder columns for good rows (same as for bad rows)
receipts_2_df = receipts_2_df.select(
    "Record_Type",        # Ensure Record_Type comes first
    "Company_Code",       # Ensure Company_Code is second
    "Receipt_Reference",  # Ensure Receipt_Reference comes third
    *[col for col in receipts_2_df.columns if col not in ["Record_Type", "Receipt_Reference", "Company_Code"]]  # Add all other columns except the ones already specified
)

# Go back to level 1 GOOD and remove any references that are bad at level 2
receipts_1_df = receipts_1_df.join(
    receipts_2_bad_df.select("Receipt_Reference"),  # Select bad Receipt_Reference from level 2
    on="Receipt_Reference",
    how="left_anti"  # Perform left anti join to remove bad Receipt_References from level 1 good
)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 21, Finished, Available, Finished)

In [21]:
if debug:
    display(receipts_2_bad_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 22, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a79d7a99-053f-43eb-a45e-c1c702327f3d)

# Init Error Check Process Level 3

In [22]:
receipts_3_Mandatory_Columns = [
    "Receipt_Date",
    "Receipt_Reference",
    "Product_Code",
    "Invoice_Key",
    "Record_Type",
    "Item_Value",
    "`NOTC(a)`",
    "Invoiced_Quantity",
    "Quantity_Code",
    "Preference_Document_type",
    "Country_of_Origin",
    "Country_of_Consignment",
    "Received_Quantity",
    "Receipt_Type",
    "Project_Reference",
    "Company_Code"
  ]

receipts_3_Cant_Be_Zero_Columns = [
    "Received_Quantity"
]

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 23, Finished, Available, Finished)

In [23]:
level = 3

null_condition = None
null_column_names_exprs = []

zero_condition = None
zero_column_names_exprs = []

# Build null checks and collect column names
for column in receipts_3_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Build "can't be zero" checks and collect column names
for column in receipts_3_Cant_Be_Zero_Columns:
    condition = col(column) == "0.000"
    zero_condition = condition if zero_condition is None else zero_condition | condition
    zero_column_names_exprs.append(when(condition, lit(column)))

# Get bad Receipt_Reference + error_fields from previous levels
load_id_col = "Receipt_Reference"
bad_level1_ids = receipts_1_bad_df.select(load_id_col, col("error_fields").alias("level1_errors"))
bad_level2_ids = receipts_2_bad_df.select(load_id_col, col("error_fields").alias("level2_errors"))

# Join level 3 with previous level error references
receipts_3_with_join = receipts_3_df \
    .join(bad_level1_ids, on=load_id_col, how="left") \
    .join(bad_level2_ids, on=load_id_col, how="left")

# Track null failures
receipts_3_with_errors = receipts_3_with_join.withColumn(
    "level3_null_failed_columns", array(*null_column_names_exprs)
).withColumn(
    "level3_null_failed_columns",
    expr("filter(level3_null_failed_columns, x -> x is not null)")
).withColumn(
    "level3_null_errors",
    when(size(col("level3_null_failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(", ", col("level3_null_failed_columns")), lit(f" are null at level {level}")))
)

# Track zero failures
receipts_3_with_errors = receipts_3_with_errors.withColumn(
    "level3_zero_failed_columns", array(*zero_column_names_exprs)
).withColumn(
    "level3_zero_failed_columns",
    expr("filter(level3_zero_failed_columns, x -> x is not null)")
).withColumn(
    "level3_zero_errors",
    when(size(col("level3_zero_failed_columns")) > 0,
         concat_ws("", concat_ws(", ", col("level3_zero_failed_columns")), lit(f" start & end with 0 at level {level}")))
)

# Combine level 3 errors
receipts_3_with_errors = receipts_3_with_errors.withColumn(
    "level3_error_fields",
    concat_ws(" , ", col("level3_null_errors"), col("level3_zero_errors"))
)

# Combine all error messages
receipts_3_with_errors = receipts_3_with_errors.withColumn(
    "error_fields",
    concat_ws("; ", array_distinct(
        split(concat_ws("; ",
            *[col(c) for c in ["level1_errors", "level2_errors", "level3_error_fields"] if c in receipts_3_with_errors.columns]
        ), "; ")
    ))
)

# Split bad and good
receipts_3_bad_df = receipts_3_with_errors.filter(
    null_condition | zero_condition | col("level1_errors").isNotNull() | col("level2_errors").isNotNull()
).drop(
    "level1_errors", "level2_errors", "level3_null_failed_columns", "level3_zero_failed_columns",
    "level3_null_errors", "level3_zero_errors", "level3_error_fields"
)

receipts_3_df = receipts_3_with_errors.filter(
    ~(null_condition | zero_condition | col("level1_errors").isNotNull() | col("level2_errors").isNotNull())
).drop(
    "level1_errors", "level2_errors", "level3_null_failed_columns", "level3_zero_failed_columns",
    "level3_null_errors", "level3_zero_errors", "level3_error_fields", "error_fields"
)

# Remove level 3 bads from earlier levels
receipts_1_df = receipts_1_df.join(
    receipts_3_bad_df.select("Receipt_Reference"),
    on="Receipt_Reference",
    how="left_anti"
)

receipts_2_df = receipts_2_df.join(
    receipts_3_bad_df.select("Receipt_Reference"),
    on="Receipt_Reference",
    how="left_anti"
)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 24, Finished, Available, Finished)

# Reorder columns after transformations

In [24]:
dataframes = [receipts_1_df, receipts_2_df, receipts_3_df, receipts_2_bad_df, receipts_3_bad_df]

# fixed column order
fixed_column_order = ["Record_Type", "Company_Code", "Receipt_Reference"]

# Reorder columns for each dataframe
receipts_1_df = receipts_1_df.select(
    *fixed_column_order,  # Ensure these columns are first
    *[col(c) for c in receipts_1_df.columns if c not in fixed_column_order]  # Keep the remaining columns
)
receipts_2_df = receipts_2_df.select(
    *fixed_column_order,  # Ensure these columns are first
    *[col(c) for c in receipts_2_df.columns if c not in fixed_column_order]  # Keep the remaining columns
)
receipts_3_df = receipts_3_df.select(
    *fixed_column_order,  # Ensure these columns are first
    *[col(c) for c in receipts_3_df.columns if c not in fixed_column_order]  # Keep the remaining columns
)
receipts_2_bad_df = receipts_2_bad_df.select(
    *fixed_column_order,  # Ensure these columns are first
    *[col(c) for c in receipts_2_bad_df.columns if c not in fixed_column_order]  # Keep the remaining columns
)
receipts_3_bad_df = receipts_3_bad_df.select(
    *fixed_column_order,  # Ensure these columns are first
    *[col(c) for c in receipts_3_bad_df.columns if c not in fixed_column_order]  # Keep the remaining columns
)

if debug:
    display(receipts_1_df)
    display(receipts_2_df)
    display(receipts_3_df)
    display(receipts_1_bad_df)
    display(receipts_2_bad_df)
    display(receipts_3_bad_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 25, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3e4b653f-4e9d-42b8-83d0-057102fb503d)

SynapseWidget(Synapse.DataFrame, 8ed05b0a-4967-4b62-a529-e564e183d66f)

SynapseWidget(Synapse.DataFrame, 830c9602-18bc-41f9-b7d7-edb69bb55864)

SynapseWidget(Synapse.DataFrame, 20a944ec-7dee-4daa-949c-75d5935ab855)

SynapseWidget(Synapse.DataFrame, bc3f4d9b-53fc-4ede-b9fe-81a2e789b35d)

SynapseWidget(Synapse.DataFrame, 114c1932-1583-4976-8c7f-5761e2839b89)

In [25]:
receipts_1_bad_keys = [row["Receipt_Reference"] for row in receipts_1_bad_df.select("Receipt_Reference").distinct().collect()]
receipts_2_bad_keys = [row["Receipt_Reference"] for row in receipts_2_bad_df.select("Receipt_Reference").distinct().collect()]
receipts_3_bad_keys = [row["Receipt_Reference"] for row in receipts_3_bad_df.select("Receipt_Reference").distinct().collect()]

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 26, Finished, Available, Finished)

In [26]:
if debug:
    print("Level 1 Bad: " , receipts_1_bad_keys)
    print("Level 2 Bad: " , receipts_2_bad_keys)
    print("Level 3 Bad: " , receipts_3_bad_keys)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 27, Finished, Available, Finished)

Level 1 Bad:  ['LD-3602532', 'LD-3989947', 'LD-3989949', 'LD-3989945', 'LD-3825365', 'LD-3989948', 'LD-3798936', 'LD-3989946', 'LD-3989941', 'LD-3989942', 'LD-3990001']
Level 2 Bad:  ['LD-3602532', 'LD-3989947', 'LD-3989949', 'LD-3989945', 'LD-3825365', 'LD-3989948', 'LD-3798936', 'LD-3989946', 'LD-3989941', 'LD-3989942', 'LD-3990001']
Level 3 Bad:  ['LD-3602532', 'LD-3825365', 'LD-3989947', 'LD-3989949', 'LD-3989945', 'LD-3989948', 'LD-3798936', 'LD-3989946', 'LD-3989941', 'LD-3989942', 'LD-3990001']


# Before Relational Bad Row Removal

In [27]:
if debug:
    display(receipts_1_bad_df)
    display(receipts_2_bad_df)
    display(receipts_3_bad_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 28, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 70f91feb-306d-40ef-8f67-e75b25452719)

SynapseWidget(Synapse.DataFrame, ffbc5cc5-2a70-4949-9fe1-1528d400ab2f)

SynapseWidget(Synapse.DataFrame, 2f491f33-3e88-4eb1-b75c-a85b9adc9894)

In [28]:
receipts_2_df = receipts_2_df.filter(
    ~col("Receipt_Reference").isin(receipts_1_bad_keys)
)

receipts_2_df = receipts_2_df.filter(
    ~col("Receipt_Reference").isin(receipts_3_bad_keys)
)


receipts_3_df = receipts_3_df.filter(
    ~col("Receipt_Reference").isin(receipts_1_bad_keys)
)

receipts_3_df = receipts_3_df.filter(
    ~col("Receipt_Reference").isin(receipts_2_bad_keys)
)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 29, Finished, Available, Finished)

## After Relational Bad Row Removal

In [29]:
if debug:
    display(receipts_1_bad_df)
    display(receipts_2_bad_df)
    display(receipts_3_bad_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 30, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5f14ee4e-7f08-4a4e-92c4-7e535197bac9)

SynapseWidget(Synapse.DataFrame, b41dd518-2f6a-4991-853b-896cf9e5028a)

SynapseWidget(Synapse.DataFrame, e5ac77b9-2ae7-459c-b17b-d7294630928d)

## If it's in Level_1 bad, then is it in in Level_2 & Level_3 good (it should NOT be)

In [30]:
if debug:
    display(receipts_1_bad_df)
    display(receipts_2_df)
    display(receipts_3_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 31, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5e7c4959-9368-4bcd-8cca-79339f6137c8)

SynapseWidget(Synapse.DataFrame, 7defb9d8-9764-43d3-a465-4ac0873a703a)

SynapseWidget(Synapse.DataFrame, fe9535ad-8a2f-40f6-85bc-aafe1512bf79)

## If it's in Level_2 bad, then is it in in Level_1 & Level_3 good (it should NOT be)

In [31]:
if debug:
    display(receipts_2_bad_df)
    display(receipts_1_df)
    display(receipts_3_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 32, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 73a20f01-7cb3-4a6d-b11e-a1446c4ea5a8)

SynapseWidget(Synapse.DataFrame, a6fba358-7fc1-45ca-9447-1c6e308c553c)

SynapseWidget(Synapse.DataFrame, 3d6b1f70-e96a-4a11-9908-da3dd14d821b)

## If it's in Level_3 bad, then is it in in Level_1 & Level_2 good (it should NOT be)

In [32]:
if debug:
    display(receipts_3_bad_df)
    display(receipts_1_df)
    display(receipts_2_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 33, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2dee2efa-cce2-4768-95d2-b03a79146686)

SynapseWidget(Synapse.DataFrame, 9973d045-f9bb-45e7-9326-3d02e029429c)

SynapseWidget(Synapse.DataFrame, 359fb3c3-f7f2-4ecf-acf2-013015246d93)

# Init Good File Name & Date Logic

In [33]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "receipts" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

file_name_1 = "receipts_1" + "_" + file_datetime + file_extention
file_path_1 = file_path_folder + file_name_1

file_name_2 = "receipts_2" + "_" + file_datetime + file_extention
file_path_2 = file_path_folder + file_name_2

file_name_3 = "receipts_3" + "_" + file_datetime + file_extention
file_path_3 = file_path_folder + file_name_3

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 34, Finished, Available, Finished)

In [34]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)
    print("Good File Name_1: " , file_name_1)
    print("Good File Name_1: " , file_path_1)
    print("Good File Name_2: " , file_name_2)
    print("Good File Name_2: " , file_path_2)
    print("Good File Name_3: " , file_name_3)
    print("Good File Name_3: " , file_path_3)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 35, Finished, Available, Finished)

Now:  2025-05-09 08:47:08.243105
Cutoff:  2025-05-09 17:15:00
File Date:  2025-05-09-08
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  receipts_2025-05-09-08.dat
Good File Name:  /lakehouse/default/Files/Output/receipts_2025-05-09-08.dat
Good File Name_1:  receipts_1_2025-05-09-08.dat
Good File Name_1:  /lakehouse/default/Files/Output/receipts_1_2025-05-09-08.dat
Good File Name_2:  receipts_2_2025-05-09-08.dat
Good File Name_2:  /lakehouse/default/Files/Output/receipts_2_2025-05-09-08.dat
Good File Name_3:  receipts_3_2025-05-09-08.dat
Good File Name_3:  /lakehouse/default/Files/Output/receipts_3_2025-05-09-08.dat


# Init Good Row Merge

In [35]:
file_name = "receipts" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

if debug:
    print("file: ", file_name, " path: " , file_path)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 36, Finished, Available, Finished)

file:  receipts_2025-05-09-08.dat  path:  /lakehouse/default/Files/Output/receipts_2025-05-09-08.dat


In [36]:
if incremental_run:

    # THIS IS ONLY FOR RECORD TRACKING BEFORE KEY COLUMNS ARE DROPPED
    
    def flatten_and_combine(df, Receipt_Reference):
        """Concats all column values into a single string, adds a timestamp, Key column, and Receipt_Reference column."""
        key_column = df.columns[-1]  # Assuming 'Key' is the last column
    
        return df.withColumn("Timestamp", F.current_timestamp()) \
                .withColumn("CombinedText", F.concat_ws("|", *df.columns[:-1])) \
                .withColumn("Receipt_Reference", F.col(Receipt_Reference)) \
                .select("Timestamp", key_column, "Receipt_Reference", "CombinedText")
    
    # Apply to each DataFrame with the appropriate Receipts_Reference column
    df1_flat = flatten_and_combine(receipts_1_df, "Receipt_Reference")
    df2_flat = flatten_and_combine(receipts_2_df, "Receipt_Reference")
    df3_flat = flatten_and_combine(receipts_3_df, "Receipt_Reference")
    
    # Combine them all
    combined_df = df1_flat.unionByName(df2_flat).unionByName(df3_flat)

    if debug: 
        display(combined_df)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 37, Finished, Available, Finished)

In [37]:
if incremental_run:

    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)
    
    lakehouse_table_name = "bondedwarehouserecordtracking_receipts"
    container_column = "Receipt_Reference"
    
    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()
    
        # Filter each input dataframe to EXCLUDE already sent
        receipts_1_df = receipts_1_df.join(sentrecords_df, receipts_1_df["Receipt_Reference"] == sentrecords_df["Receipt_Reference"], "left_anti")
        receipts_2_df = receipts_2_df.join(sentrecords_df, receipts_2_df["Receipt_Reference"] == sentrecords_df["Receipt_Reference"], "left_anti")
        receipts_3_df = receipts_3_df.join(sentrecords_df, receipts_3_df["Receipt_Reference"] == sentrecords_df["Receipt_Reference"], "left_anti")
    
        # flatten and combine to add current rows to record tracking table later
        df1_flat = flatten_and_combine(receipts_1_df, "Receipt_Reference")
        df2_flat = flatten_and_combine(receipts_2_df, "Receipt_Reference")
        df3_flat = flatten_and_combine(receipts_3_df, "Receipt_Reference")
    
        combined_df = df1_flat.unionByName(df2_flat).unionByName(df3_flat)
    
    except Exception as e:
        print(f"An error occurred: {e}")


StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 38, Finished, Available, Finished)

# Export Good

In [38]:
save_dataframe_to_csv(receipts_1_df.drop("Key","Level_Key","Parent_Key"), file_path_1, show_header=False)
save_dataframe_to_csv(receipts_2_df.drop("Key","Level_Key","Parent_Key"), file_path_2, show_header=False)
save_dataframe_to_csv(receipts_3_df.drop("Key","Level_Key","Parent_Key"), file_path_3, show_header=False)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 39, Finished, Available, Finished)

Add pipes: True
Show headers: False
Add pipes: True
Show headers: False
Add pipes: True
Show headers: False


In [39]:
dat_file_paths = [
    file_path_1,
    file_path_2,
    file_path_3
]
if debug:
    list(dat_file_paths)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 40, Finished, Available, Finished)

In [40]:
try:
    receipts_1_dat = pd.read_csv(file_path_1, sep='|', header=None, dtype=str)
    receipts_2_dat = pd.read_csv(file_path_2, sep='|', header=None, dtype=str)
    receipts_3_dat = pd.read_csv(file_path_3, sep='|', header=None, dtype=str)
    good_rows = True

except:
    print("Empty file here at GOOD receipts level")
    good_rows = False
    ready_to_copy = False

    try:
        for file in dat_file_paths:
            os.remove(file)
            print(f"Deleted: {file}")
    except:
        print("Tried to delete part files, and failed.")

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 41, Finished, Available, Finished)

Empty file here at GOOD receipts level
Deleted: /lakehouse/default/Files/Output/receipts_1_2025-05-09-08.dat
Deleted: /lakehouse/default/Files/Output/receipts_2_2025-05-09-08.dat
Deleted: /lakehouse/default/Files/Output/receipts_3_2025-05-09-08.dat


In [41]:
if good_rows:
    
    receipts_123_dat = pd.concat([receipts_1_dat, receipts_2_dat, receipts_3_dat], ignore_index=True)
    receipts_123_dat = receipts_123_dat.sort_values(by=[2, 1]).reset_index(drop=True)

    receipts_123_dat.loc[receipts_123_dat[0] == '3|ENDCTG|', 1] = ''

    # Convert the DataFrame back to CSV format
    csv_data = receipts_123_dat.to_csv(sep='|', index=False, header=False)

    # spoof site column
    csv_data = csv_data.replace("3|ENDCTG|", "3|ENDCTG||")

    display(csv_data)
    # Write the cleaned CSV data back to the file
    with open(file_path, 'w') as file:
        file.write(csv_data)

    if debug:
        display(receipts_123_dat)
    
    ready_to_copy = True

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 42, Finished, Available, Finished)

In [42]:
if good_rows:
    
    for file in dat_file_paths:
        os.remove(file)
        print(f"Deleted: {file}")

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 43, Finished, Available, Finished)

# Init Bad File Name

In [43]:
error_file_1 = "receipts_1_errors_" + file_datetime + file_extention
error_file_path_1 = file_path_folder + error_file_1

error_file_2 = "receipts_2_errors_" + file_datetime + file_extention
error_file_path_2 = file_path_folder + error_file_2

error_file_3 = "receipts_3_errors_" + file_datetime + file_extention
error_file_path_3 = file_path_folder + error_file_3

if debug:
    print("file: ", error_file_1, " path: " , error_file_path_1)
    print("file: ", error_file_2, " path: " , error_file_path_2)
    print("file: ", error_file_3, " path: " , error_file_path_3)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 44, Finished, Available, Finished)

file:  receipts_1_errors_2025-05-09-08.dat  path:  /lakehouse/default/Files/Output/receipts_1_errors_2025-05-09-08.dat
file:  receipts_2_errors_2025-05-09-08.dat  path:  /lakehouse/default/Files/Output/receipts_2_errors_2025-05-09-08.dat
file:  receipts_3_errors_2025-05-09-08.dat  path:  /lakehouse/default/Files/Output/receipts_3_errors_2025-05-09-08.dat


# Export Bad

In [44]:
save_dataframe_to_csv(receipts_1_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_1, show_header = True)
save_dataframe_to_csv(receipts_2_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_2, show_header = True)
save_dataframe_to_csv(receipts_3_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_3, show_header = True)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 45, Finished, Available, Finished)

Add pipes: True
Show headers: True
Add pipes: True
Show headers: True
Add pipes: True
Show headers: True


In [45]:
# Read the saved CSV file into memory as a string
with open(error_file_path_3, "r") as file:
    csv_content = file.read()

# Spoof extra pipe for site column in level 3 error file
csv_content_modified = csv_content.replace("3|ENDCTG|", "3|ENDCTG||")

# Write the modified content back to the same file
with open(error_file_path_3, "w") as file:
    file.write(csv_content_modified)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 46, Finished, Available, Finished)

# Init Record Tracking

In [46]:
if incremental_run == True and good_rows == True:
    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(combined_df, f"{table_prefix}receipts", file_name)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 47, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [49]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, 50, Finished, Available, Finished)

ExitValue: Process Complete

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [ ]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, -1, Cancelled, , Cancelled)

In [ ]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, -1, Cancelled, , Cancelled)

In [ ]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, -1, Cancelled, , Cancelled)

In [ ]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, 704fb301-05bf-436a-95ad-13eb215e6e5f, -1, Cancelled, , Cancelled)